This code contains three parts:
- calculation of PDFs for ThxOy clusters
- calculation of PDFs for experimental crystal structures from CSD
- preparation of experimental PDFs for evaluation by the trained model

In [ ]:
from diffpy.Structure import loadStructure
from diffpy.srreal.pdfcalculator import DebyePDFCalculator
from pyobjcryst import loadCrystal
import glob
import numpy as np
import os
import fnmatch
import pandas as pd

Calculation of PDFs for ThxOy clusters:

In [ ]:
os.chdir('/your/project/directory/with/model/clusters/')
files = glob.glob('*.xyz')

In [ ]:
# Calculate the correct number of atoms in each xyz file
for f in files:
    with open(f, "r") as file:
        contents = file.readlines()
        new_first_line = str(len(contents) - 2) + "\n"
        contents[0] = new_first_line
    with open(f, "w") as file:
        file.writelines(contents)

In [ ]:
for f in files:
    model = loadStructure(f)
    dpc = DebyePDFCalculator()
    dpc.qmax = 11
    dpc.rmax = 20
    r, g = dpc(model, qmin=0.3)
    datagcalc = np.column_stack([r, g])
    np.savetxt(f.replace('.xyz','.dat'), datagcalc, header='r g')

Calculation of PDFs for experimental crystal structures from CSD:

In [ ]:
os.chdir('/your/project/directory/with/cifs/')
files = glob.glob('*.cif')

In [ ]:
i = 0
for f in files:
    # CIF files loaded from CSD may be wrongly formatted or corrupted which would make them 
    # not accessible for DiffPy, we will therefore iterate over the files using try-except blocks
    try:
        model = loadCrystal(f)
        dpc = DebyePDFCalculator()
        dpc.qmax = 11
        dpc.rmax = 20
        r, g = dpc(model, qmin=0.3)
        datagcalc = np.column_stack([r, g])
        np.savetxt(f.replace('.cif', '.dat'), datagcalc, header='r g')
    except Exception as e:
        print(f"An error occurred while processing file {f}: {str(e)}")
        continue
    i += 1

Experimental PDF data may be recorded with uneven distance steps, therefore the interpolation is required to make the experimental PDF data fully consistent with the calculated PDFs the model was trianed on. Preparation of experimental PDFs for evaluation by the trained model:

In [ ]:
os.chdir('/your/project/directory/with/experimental/PDFs')
def clean_files(pattern):
    for file in os.listdir('.'):
        if fnmatch.fnmatch(file, pattern):
            file_path = os.path.join('.', file)
            try:
                if os.path.isfile(file_path):
                    os.unlink(file_path)
            except Exception as e:
                print(f'Failed to delete {file_path}. Reason: {e}')

clean_files('*processed*')
files = glob.glob('*.gr')

In [ ]:
for f in files:
    data = pd.read_csv(f, header=29, delim_whitespace=True)
    df = pd.DataFrame(data)
    data.columns = ['r', 'g(r)']
    df = df.loc[(df["r"] >= 2) & (df["r"] <= 12)]
    x = np.arange(2, 12.01, 0.01)
    df["interpolated_data"] = np.interp(x, df["r"], df["g(r)"])
    df = df.drop(columns=['g(r)'])
    df.rename(columns={"interpolated_data":'g(r)'}, inplace=True)
    df.to_csv(f.replace('.gr', '_processed.gr'), index=False, sep='\t')